# Stage 1: acceptance audit, Qwen3-0.6B drafting for Qwen3-1.7B

Measures how often the draft model's guesses would be accepted by the target, on all 1,012 FLORES-200 devtest sentences in Urdu and English. All logic lives in the `zubaan` package; this notebook only calls the CLI.

Settings before running:

- Accelerator: **GPU T4** (one GPU is enough) or P100.
- Internet: **on**.
- No secret is needed: Qwen3 and FLORES-200 (Meta archive) are not gated. If you switch to FLORES+ (`--source hf`), add `HF_TOKEN` under Add-ons > Secrets.

Both models run in **float32** (about 9 GB of the 16 GB GPU): T4 and P100 have no bfloat16, and float32 avoids float16 overflow. Expected run time: under 30 minutes.

In [ ]:
# Install zubaan from GitHub (pin a commit for reproducibility by appending @<commit>).
REPO = "git+https://github.com/Hashir-Aziz123/zubaan.git"
!pip install -q "zubaan[audit] @ {REPO}"
!zubaan version

In [ ]:
import torch

assert torch.cuda.is_available(), "Turn on a GPU accelerator in the notebook settings"
print(torch.cuda.get_device_name(0))

In [ ]:
!zubaan data flores --out data

In [ ]:
# Self-test first: a model against itself must give alpha = 1.000.
SELF = "-d Qwen/Qwen3-0.6B -t Qwen/Qwen3-0.6B"
PAIR = "-d Qwen/Qwen3-0.6B -t Qwen/Qwen3-1.7B"
DATA = "-c data/flores_urd.txt -r data/flores_eng.txt"
GPU = "--dtype float32 --device cuda"
OUT = "/kaggle/working"
!zubaan audit acceptance {SELF} {DATA} {GPU} --max-texts 5 --out {OUT}/selftest.json

In [ ]:
!zubaan audit acceptance {PAIR} {DATA} {GPU} --max-texts 1012 --out {OUT}/acceptance_qwen3.json

Download `acceptance_qwen3.json` from the Output tab into your local `reports/` folder, then build the report with `zubaan report tax`.